# Rossmann Sales Forecasting
## 03 — Feature Engineering

### 1. Load and merge

- Combined training and test records so both receive the same feature definitions.
- Excluded `Customers`, which is unavailable when predicting future sales, and merged store metadata using `Store`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

raw_dir = Path("../data/raw")
train = pd.read_csv(raw_dir / "train.csv", parse_dates=["Date"], dtype={"StateHoliday": "string"})
test = pd.read_csv(raw_dir / "test.csv", parse_dates=["Date"], dtype={"StateHoliday": "string"})
store = pd.read_csv(raw_dir / "store.csv")

train["Split"] = "train"
test["Split"] = "test"

records = pd.concat(
    [train.drop(columns="Customers"), test.drop(columns="Id")],
    ignore_index=True
)
records = records.merge(store, on="Store", how="left", validate="many_to_one")
records = records.sort_values(["Store", "Date"]).reset_index(drop=True)

if records["StoreType"].isna().any():
    raise ValueError("Some stores have no matching metadata")

print("Train:", train.shape, "Test:", test.shape)
print("Merged rows:", len(records))

Train: (1017209, 10) Test: (41088, 9)
Merged rows: 1058297


### 2. Calendar features

- Extracted calendar information from `Date` and identified weekends.
- Preserved an `OpenMissing` indicator before assuming missing `Open` values represent open stores.

In [2]:
records["Year"] = records["Date"].dt.year
records["Month"] = records["Date"].dt.month
records["DayOfMonth"] = records["Date"].dt.day
records["IsWeekend"] = records["DayOfWeek"].isin([6, 7]).astype(int)

records["OpenMissing"] = records["Open"].isna().astype(int)
records["Open"] = records["Open"].fillna(1).astype(int)

### 3. Time-series validation

- Created two expanding-window splits from unique calendar dates rather than individual store rows.
- Used a 48-day validation horizon for each fold.

In [3]:
HORIZON = 48
train_dates = pd.DatetimeIndex(train["Date"].unique()).sort_values()

if not train_dates.equals(pd.date_range(train_dates.min(), train_dates.max())):
    raise ValueError("Training dates are not consecutive")

splitter = TimeSeriesSplit(n_splits=2, test_size=HORIZON)
folds = []

for number, (train_idx, val_idx) in enumerate(splitter.split(train_dates), start=1):
    origin = train_dates[train_idx[-1]]
    start = train_dates[val_idx[0]]
    end = train_dates[val_idx[-1]]
    folds.append((number, origin, start, end))

display(pd.DataFrame(folds, columns=["Fold", "Origin", "Start", "End"]))

,Fold,Origin,Start,End
0,1,2015-04-26,2015-04-27,2015-06-13
1,2,2015-06-13,2015-06-14,2015-07-31


### 4. Forecast origins

- Assigned one fixed forecast origin to each 48-day training block.
- Used the final training date as the origin for test predictions and checked validation-origin alignment.

In [4]:
end_train = train["Date"].max()
days_back = (end_train - records["Date"]).dt.days
block = days_back // HORIZON + 1

records["OriginDate"] = end_train - pd.to_timedelta(block * HORIZON, unit="D")
records.loc[records["Split"] == "test", "OriginDate"] = end_train
records["HorizonDay"] = (records["Date"] - records["OriginDate"]).dt.days

if not records["HorizonDay"].between(1, HORIZON).all():
    raise ValueError("Dates fall outside the 48-day forecast blocks")

for number, origin, start, end in folds:
    rows = records[(records["Split"] == "train") & records["Date"].between(start, end)]
    if not rows["OriginDate"].eq(origin).all():
        raise ValueError(f"Incorrect forecast origin in Fold {number}")

### 5. Competition and Promo2 features

- Estimated competitor opening from the recorded month; calculated activity and elapsed months at each forecast origin.
- Calculated Promo2 start dates from ISO year/week and retained the applicable interval category.
- Preserved unknown start information rather than inventing event dates.

In [5]:
# Competition start is recorded by month, so use its first day.
competition_start = pd.to_datetime({
    "year": records["CompetitionOpenSinceYear"],
    "month": records["CompetitionOpenSinceMonth"],
    "day": 1
}, errors="coerce")

records["CompetitionOpen"] = (
    competition_start <= records["OriginDate"]
).where(competition_start.notna()).astype(float)

records["CompetitionOpenMonths"] = (
    (records["OriginDate"].dt.year - competition_start.dt.year) * 12
    + records["OriginDate"].dt.month - competition_start.dt.month
).clip(lower=0)

records.loc[records["CompetitionOpen"].eq(0), "CompetitionDistance"] = float("nan")
records["CompetitionStartMissing"] = competition_start.isna().astype(int)
records["CompetitionDistanceMissing"] = records["CompetitionDistance"].isna().astype(int)

# Promo2 starts in an ISO calendar year and week.
promo_year = records["Promo2SinceYear"].astype("Int64").astype("string")
promo_week = records["Promo2SinceWeek"].astype("Int64").astype("string").str.zfill(2)
promo_start = pd.to_datetime(
    promo_year + "-W" + promo_week + "-1",
    format="%G-W%V-%u", errors="coerce"
)

records["Promo2Started"] = (
    promo_start <= records["OriginDate"]
).astype(float)
records.loc[records["Promo2"].eq(0), "Promo2Started"] = 0
records.loc[records["Promo2"].eq(1) & promo_start.isna(), "Promo2Started"] = float("nan")

# Hide future schedules; distinguish not started from unknown.
records["PromoInterval"] = records["PromoInterval"].where(
    records["Promo2Started"].eq(1), "NoPromo2"
)
records.loc[records["Promo2Started"].isna(), "PromoInterval"] = "Unknown"
records["PromoInterval"] = records["PromoInterval"].fillna("Unknown")

### 6. Historical lags

- Expanded each store to a complete daily calendar before calculating `Lag49` and `Lag56`.
- Kept missing calendar observations as missing, not zero sales.

In [6]:
all_dates = pd.date_range(train["Date"].min(), test["Date"].max())
calendar = pd.MultiIndex.from_product(
    [store["Store"], all_dates], names=["Store", "Date"]
)

sales = train.set_index(["Store", "Date"])["Sales"].reindex(calendar)
open_days = train.set_index(["Store", "Date"])["Open"].reindex(calendar)

history = pd.DataFrame(index=calendar)
history["Lag49"] = sales.groupby(level="Store").shift(49)
history["Lag56"] = sales.groupby(level="Store").shift(56)

### 7. Frozen historical averages

- Averaged available positive sales in the final 7 or 28 **calendar days** through each forecast origin.
- Applied those fixed averages throughout the associated 48-day forecast window.

In [7]:
positive_sales = sales.where((open_days == 1) & (sales > 0))

for window in [7, 28]:
    history[f"FrozenMean{window}"] = positive_sales.groupby(level="Store").transform(
        lambda values: values.rolling(window, min_periods=1).mean()
    )

history = history.reset_index()
records = records.merge(
    history[["Store", "Date", "Lag49", "Lag56"]],
    on=["Store", "Date"], how="left", validate="many_to_one"
)

frozen = history[["Store", "Date", "FrozenMean7", "FrozenMean28"]].rename(
    columns={"Date": "OriginDate"}
)
records = records.merge(frozen, on=["Store", "OriginDate"], how="left", validate="many_to_one")

### 8. Inspect and export

- Checked row counts and training/test feature alignment before exporting the processed datasets.
- Left remaining numerical missing values for fold-specific imputation during model training.

In [8]:
drop_columns = [
    "Split", "Promo2", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
    "Promo2SinceWeek", "Promo2SinceYear"
]

train_features = records[records["Split"] == "train"].drop(columns=drop_columns).copy()
test_features = records[records["Split"] == "test"].drop(columns=drop_columns + ["Sales"]).copy()

if len(train_features) != len(train) or len(test_features) != len(test):
    raise ValueError("The number of store-day records changed")
if not train_features.drop(columns="Sales").columns.equals(test_features.columns):
    raise ValueError("Training and test columns do not match")

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)

missing = pd.DataFrame({
    "Train": train_features[test_features.columns].isna().sum(),
    "Test": test_features.isna().sum()
})
display(missing[missing.sum(axis=1) > 0])

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
train_features.to_csv(output_dir / "train_features.csv", index=False)
test_features.to_csv(output_dir / "test_features.csv", index=False)

Train features: (1017209, 28)
Test features: (41088, 27)


,Train,Test
CompetitionDistance,91876,192
CompetitionOpen,323348,15216
CompetitionOpenMonths,323348,15216
Lag49,63455,0
Lag56,72520,0
FrozenMean7,38345,0
FrozenMean28,37769,0


### Output observations

- The exported datasets contain **1,017,209 training rows** and **41,088 test rows**; the test dataset excludes the target `Sales`.
- Missing competition dates remain visible in `CompetitionOpen` and `CompetitionOpenMonths`.
- Missing historical features were retained as `NaN`, rather than being replaced with fabricated sales values.

### Modeling notes

- Historical features use only sales available at their assigned forecast origins. `Lag49` and `Lag56` also use actual calendar-day offsets.
- `Promo2Started` and `PromoInterval` represent the usable Promo2 status; raw `Promo2` is excluded to avoid revealing later participation in earlier historical blocks.
- `CompetitionOpen` uses the first of the recorded month as an approximation; unknown starts remain missing.
- Missing values will be handled by training-fold-only preprocessing in Notebook 04. Store metadata comes from one snapshot, so its availability at past origins cannot be fully verified.